In [1]:
import pickle
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax

In [2]:
class PatientModel:
    def __init__(self, diagnoses, symptoms, adj_matrix, diag_vectors, symp_vectors,
                 beta=0.4, decay=0.85, top_k=5):
        self.diagnoses = diagnoses
        self.symptoms = symptoms
        self.diag_vectors = diag_vectors
        self.symp_vectors = symp_vectors
        self.adj_matrix = adj_matrix
        self.beta = beta
        self.decay = decay
        self.top_k = top_k
        self.diag_scores = np.zeros(len(diagnoses))
        self.symp_scores = np.zeros(len(symptoms))

    def get_diagnosis_activations(self):
        return self.diag_scores

    # РЕАЛІЗАЦІЯ ФОРМУЛИ 3.5: Моделювання динаміки (Марківський процес)
    def update_state(self, action_vector, revealed_symptoms=None):
        # 1. Розрахунок схожості дії студента з симптомами
        symp_sim = cosine_similarity(action_vector.reshape(1, -1), self.symp_vectors)[0]
        sparse_symp = np.zeros_like(symp_sim)

        # Вибираємо топ-K найбільш релевантних симптомів
        top_k_idx = np.argsort(symp_sim)[::-1][:self.top_k]
        sparse_symp[top_k_idx] = symp_sim[top_k_idx]

        # Якщо пацієнт підтвердив конкретні симптоми, встановлюємо їх активацію на 1.0
        if revealed_symptoms:
            for name in revealed_symptoms:
                if name in self.symptoms:
                    idx = self.symptoms.index(name)
                    sparse_symp[idx] = 1.0

        # 2. Оновлення стану симптомів (інерція + нова дія)
        self.symp_scores = self.decay * self.symp_scores + self.beta * sparse_symp

        # 3. Трансформація стану в діагнози через матрицю суміжності W
        diag_from_symp = self.adj_matrix @ self.symp_scores

        # 4. Врахування прямої схожості дії з діагнозами
        diag_sim = cosine_similarity(action_vector.reshape(1, -1), self.diag_vectors)[0]

        # Формування нового вектора стану діагнозів St+1
        self.diag_scores = (self.decay * self.diag_scores +
                           0.7 * diag_from_symp +
                           0.3 * self.beta * diag_sim)
        return self.diag_scores

In [3]:
class VirtualPatient:
    def __init__(self, df, diagnoses, symptoms, reveal_ratio=0.3,
                 similarity_threshold=0.82, max_reveal_per_question=3):
        self.diagnoses = diagnoses
        self.symptoms = symptoms
        self.sim_threshold = similarity_threshold
        self.max_reveal = max_reveal_per_question
        self.true_diagnosis = "Localized edema" # Або завантаження з df

    def answer_question(self, question):
        # Логіка порівняння через BioBERT (Формули 3.1, 3.3) [cite: 231, 239]
        return "Yes/No", []

# --- 2. ЗАВАНТАЖЕННЯ МОДЕЛЕЙ ТА АРТЕФАКТІВ ---

MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Тепер обидва класи будуть розпізнані
with open('patient_model_class.pkl', 'rb') as f:
    patient_model = pickle.load(f)

with open('virtual_patient_class.pkl', 'rb') as f:
    virtual_patient = pickle.load(f)

print(f"Систему ініціалізовано. Пацієнт та Модель готові до Step 4.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Систему ініціалізовано. Пацієнт та Модель готові до Step 4.


In [4]:
# 1. Функція семантичної векторизації (Формула 3.1) [cite: 231, 254]
def encode_text(text: str) -> np.ndarray:
    # Використовує BioBERT для перетворення медичного тексту у вектор [cite: 255-256]
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=64, padding=True).to(device)
    with torch.no_grad():
        out = model(**inputs)
    # Повертає вектор сутності в Rd [cite: 230, 253]
    return out.last_hidden_state[:, 0, :].cpu().numpy().squeeze()

# 2. Дані для оцінювання компетенцій (Розділ 3.4) [cite: 283-284]
# Наповнення бази для порівняння з еталонами
eval_data = {
    "Localized edema": {
        # Вектори "Золотого стандарту" для розрахунку R (Формула 3.9) [cite: 299]
        "gold_standard": [
            encode_text("Apply cold compress and elevate the limb"),
            encode_text("Prescribe non-steroidal anti-inflammatory drugs"),
            encode_text("Perform lymphatic drainage massage")
        ],
        # Вектори протипоказань для розрахунку Penalty (Формула 3.10) [cite: 303-304]
        "red_flags": [
            encode_text("Apply intense heat to the swelling"),
            encode_text("Administer aggressive deep tissue massage"),
            encode_text("Prescribe medication that increases fluid retention")
        ]
    }
}

print("Базу еталонних протоколів успішно векторизовано та завантажено.")

Базу еталонних протоколів успішно векторизовано та завантажено.


In [5]:
def calculate_step_metrics(action_text, true_diagnosis, delta_h):
    """
    Реалізація формул 3.9 - 3.11
    """
    v_at = encode_text(action_text)

    # 1. Семантична коректність R (Формула 3.9) [cite: 299]
    ideals = eval_data.get(true_diagnosis, {}).get("gold_standard", [])
    r_score = 0.0
    if ideals:
        r_score = max([cosine_similarity(v_at.reshape(1,-1), v.reshape(1,-1))[0][0] for v in ideals])

    # 2. Штраф за помилки Penalty (Формула 3.10) [cite: 304]
    red_flags = eval_data.get(true_diagnosis, {}).get("red_flags", [])
    penalty = 0.0
    if red_flags:
        p_sims = [cosine_similarity(v_at.reshape(1,-1), v.reshape(1,-1))[0][0] for v in red_flags]
        max_p = max(p_sims)
        if max_p > 0.82:  # Поріг чутливості до помилок
            penalty = 1.5 * max_p  # w_side = 1.5 [cite: 306]

    # 3. Підсумковий бал за крок (Формула 3.11) [cite: 310]
    # w1 = 0.6 (діагностика), w2 = 0.4 (протокол)
    step_score = (0.6 * delta_h) + (0.4 * r_score) - penalty

    return {
        "r_score": round(float(r_score), 4),
        "penalty": round(float(penalty), 4),
        "total_step": round(float(step_score), 4)
    }

In [6]:
# 1. Розрахунок ймовірностей (Формула 3.7) [cite: 290-291]
def compute_probabilities(activations: np.ndarray, tau: float = 1.0) -> np.ndarray:
    # Використовує нормалізуючу функцію Softmax з урахуванням температури tau [cite: 291, 296]
    return softmax(activations / tau)

# 2. Ентропійна міра Шеннона (Формула 3.6) [cite: 286-287]
def shannon_entropy(probs: np.ndarray) -> float:
    # Кількісне вимірювання ступеня невизначеності клінічної ситуації [cite: 286]
    probs = np.clip(probs, 1e-10, 1.0)
    return float(-np.sum(probs * np.log2(probs)))

# 3. Графік "охолодження" температури (Розділ 3.4)
def get_tau(step: int, tau_start: float = 1.2, tau_end: float = 0.35, n_steps: int = 10) -> float:
    # Регулює "впевненість" моделі в ході сценарію
    alpha = (tau_start - tau_end) / n_steps
    return max(tau_end, tau_start - alpha * step)

# 4. Функція для кодування дій (Формула 3.1) [cite: 231]
def embed_action(action: str) -> np.ndarray:
    # Використовує ідентичний алгоритм Encoder для дій користувача [cite: 259-260]
    # (Викликає функцію encode_text, яку ви вже визначили раніше)
    return encode_text(action)

print("Математичні функції (3.6 - 3.8) успішно оголошено.")

Математичні функції (3.6 - 3.8) успішно оголошено.


In [7]:
# Тестовий сценарій
true_diag = "Localized edema"
session_history = []
final_grade = 0.0

# Імітація дій студента
student_actions = [
    {"text": "Is there any swelling in your legs?", "dH": 0.45}, # Корисне питання
    {"text": "I will apply a hot compress to the area.", "dH": -0.1}, # Помилка (Penalty)
    {"text": "Keep your leg elevated and rest.", "dH": 0.2} # Відповідність протоколу (R)
]

print(f"{'='*60}")
print(f"ОЦІНЮВАННЯ КОМПЕТЕНЦІЙ СТУДЕНТА (Step 4)")
print(f"{'='*60}\n")

for i, act in enumerate(student_actions):
    metrics = calculate_step_metrics(act["text"], true_diag, act["dH"])
    session_history.append(metrics)
    final_grade += metrics["total_step"]

    print(f"Крок {i+1}: '{act['text']}'")
    print(f"  ΔH (Інф. прибуток): {act['dH']}")
    print(f"  R (Коректність):    {metrics['r_score']}")
    print(f"  Penalty (Штраф):    {metrics['penalty']}")
    print(f"  Бал за крок:        {metrics['total_step']}\n")

print(f"{'='*60}")
print(f"ФІНАЛЬНИЙ РЕЗУЛЬТАТ (Final Score): {round(final_grade, 4)}")
print(f"{'='*60}")

ОЦІНЮВАННЯ КОМПЕТЕНЦІЙ СТУДЕНТА (Step 4)

Крок 1: 'Is there any swelling in your legs?'
  ΔH (Інф. прибуток): 0.45
  R (Коректність):    0.8752
  Penalty (Штраф):    1.3007
  Бал за крок:        -0.6807

Крок 2: 'I will apply a hot compress to the area.'
  ΔH (Інф. прибуток): -0.1
  R (Коректність):    0.9154
  Penalty (Штраф):    1.3531
  Бал за крок:        -1.0469

Крок 3: 'Keep your leg elevated and rest.'
  ΔH (Інф. прибуток): 0.2
  R (Коректність):    0.8884
  Penalty (Штраф):    1.2986
  Бал за крок:        -0.8233

ФІНАЛЬНИЙ РЕЗУЛЬТАТ (Final Score): -2.5509


In [8]:
def integrated_diagnostic_session(student_input_list):
    total_score = 0.0
    true_diagnosis = virtual_patient.true_diagnosis

    # Початковий стан ентропії (Формула 3.6) [cite: 286-287]
    current_h = shannon_entropy(compute_probabilities(patient_model.get_diagnosis_activations(), get_tau(0)))

    print(f"{'='*30} ПОЧАТОК ІСПИТУ {'='*30}")

    for i, action_text in enumerate(student_input_list):
        # --- КРОК А: Реакція пацієнта (Step 3) ---
        # Пацієнт відповідає на питання
        answer, newly_revealed = virtual_patient.answer_question(action_text)

        # --- КРОК Б: Оновлення математичної моделі (Step 3) ---
        # Формула 3.5: Марківський перехід стану [cite: 269-270]
        revealed_names = [s for s, _ in newly_revealed]
        a_vec = embed_action(action_text)
        patient_model.update_state(a_vec, revealed_symptoms=revealed_names)

        # Обчислюємо нову ентропію та Delta H (Формула 3.8) [cite: 297-298]
        new_h = shannon_entropy(compute_probabilities(patient_model.get_diagnosis_activations(), get_tau(i+1)))
        delta_h = current_h - new_h
        current_h = new_h # оновлюємо для наступного кроку

        # --- КРОК В: Розрахунок компетенцій (Step 4) ---
        # Реалізація формул 3.9 (R), 3.10 (Penalty) та 3.11 (Final Score) [cite: 299-310]
        metrics = calculate_step_metrics(action_text, true_diagnosis, delta_h)
        total_score += metrics['total_step']

        print(f"Крок {i+1}: Студент: '{action_text}'")
        print(f"  Пацієнт: {answer}")
        print(f"  Метрики: ΔH={delta_h:.3f}, R={metrics['r_score']:.3f}, Penalty={metrics['penalty']:.3f}")
        print(f"  Бал: {metrics['total_step']:.3f}\n")

    print(f"{'='*30} ПІДСУМОК {'='*30}")
    print(f"Фінальна оцінка за сесію: {total_score:.4f}")

# Запуск тесту
integrated_diagnostic_session([
    "Do you have chest pain?",
    "I'll give you a hot massage", # Тест штрафу (3.10) [cite: 303-304]
    "Elevate your legs"            # Тест протоколу (3.9) [cite: 299-300]
])

============================== ПОЧАТОК ІСПИТУ ==============================
Крок 1: Студент: 'Do you have chest pain?'
  Пацієнт: Yes/No
  Метрики: ΔH=0.243, R=0.000, Penalty=0.000
  Бал: 0.146

Крок 2: Студент: 'I'll give you a hot massage'
  Пацієнт: Yes/No
  Метрики: ΔH=0.304, R=0.000, Penalty=0.000
  Бал: 0.183

Крок 3: Студент: 'Elevate your legs'
  Пацієнт: Yes/No
  Метрики: ΔH=0.277, R=0.000, Penalty=0.000
  Бал: 0.166

============================== ПІДСУМОК ==============================
Фінальна оцінка за сесію: 0.4947
